In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("../names.txt", "r").read().splitlines()

In [3]:
# 1. create table to map from char to index
chars = sorted(list(set("".join(words))))
stoi = {char: i+1 for i, char in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vacab_size = len(itos)
print(f"vocabulary size: {vacab_size} sample size {len(words)}")

vocabulary size: 27 sample size 32033


In [4]:
# 2. build data set with specific context window
def build_dataset(words, block_size=3):
    X = []
    Y = []
    for word in words: 
        context = [0] * block_size
        for ch in word + '.':
            index = stoi[ch]
            X.append(context)
            Y.append(index)
            context = context[1:] + [index]
    return torch.tensor(X), torch.tensor(Y)

X, Y = build_dataset(words)
print(f"X shape {X.shape} Y shape {Y.shape}")

X shape torch.Size([228146, 3]) Y shape torch.Size([228146])


In [57]:
# 3. set up MLP layers and params
g = torch.Generator().manual_seed(42)
emb_size = 10
block_size = 3

# embeddings: 27 x 2
C = torch.randn((vacab_size, emb_size), generator=g)
in_dim = emb_size * block_size
hidden_dim = 200
out_dim = vacab_size

# kaiming init
kaiming_gain = 5 / 3 #for tanh

# batch norm - trainable gain and bias
bn_gain = torch.ones(1, hidden_dim) #init value is 1
bn_bias = torch.zeros(1, hidden_dim) #init value is 0
bn_mean_running = torch.zeros((1, hidden_dim)) #running mean, no grad
bn_std_running = torch.ones((1, hidden_dim)) #running std, no grad

# layer 1: input dim: block_size * emb_size = 6
# w1: 6 x 100, b1: 100
w1 =  torch.randn((in_dim, hidden_dim), generator=g) * kaiming_gain / (in_dim ** 0.5)
b1 = torch.zeros(hidden_dim)

# layer 2: hidden dim: 100, output dim: 27
# w2: 100 x 27, b2: 27
w2 = torch.randn((hidden_dim, out_dim), generator=g) * 0.01 #small initialization
b2 = torch.zeros(out_dim)

parameters = [C, w1, b1, w2, b2, bn_gain, bn_bias]
for p in parameters:
    p.requires_grad = True

total_params = sum(p.nelement() for p in parameters)
print("Num params:", total_params)

Num params: 12297


In [24]:
# Set up training/val/test data sets
# 80% training, 10% eval, 10% test
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[: n1], block_size)
Xdev, Ydev = build_dataset(words[n1: n2], block_size)
Xtest, Ytest = build_dataset(words[n2:], block_size)


In [54]:
def batch_norm(x: torch.Tensor, gain, bias):
    mean = x.mean(dim=0, keepdim=True) # 1. Mean on batch
    var = x.var(dim=0, keepdim=True, unbiased=False)  # 2. Variance on batch
    eps = 1e-5 #epsilon
    std = torch.sqrt(var + eps) # 3. Standard deviation on batch
    norm = (x - mean) / std # 3. Normalization
    out = gain * norm + bias # 4. Scale and shift
    return out, mean, std # Also return mean and std to update running mean/std

In [58]:
# Train MLP with learning rate finder
lr_history = []
loss_history = []
loss: torch.Tensor | None = None

lre = torch.linspace(-3, 0, 20000)
lrs = 10 ** lre
sample_size = 64
for step in range(20000):
    # 1. select a mini-batch (SGD) size=32  
    batch_idxes = torch.randint(0, Xtr.shape[0], (sample_size,))

    # 2. training data with embeddings (32, 3, 2) -> (32, 6), target y (32,)
    h_input = C[Xtr[batch_idxes]].view(-1, in_dim)
    target_y = Ytr[batch_idxes]

    # 3. hidden layer with tanh activation (32, 100)
    h_output_preact = h_input @ w1 #remove b1 for batch norm
    h_output_bn, h_output_bn_mean, h_output_bn_std = batch_norm(h_output_preact, bn_gain, bn_bias)
    h_output = torch.tanh(h_output_bn) 

    # Update batch norm running mean and std
    with torch.no_grad():
        bn_mean_running = 0.999 * bn_mean_running + 0.001 * h_output_bn_mean
        bn_std_running = 0.999 * bn_std_running + 0.001 * h_output_bn_std

    # 4. output layer [32, 27]
    logits = h_output @ w2 + b2

    # 5. compute cross entropy loss
    loss = F.cross_entropy(logits, target_y)

    # 6. loss backward
    for p in parameters:
        p.grad = None
    loss.backward()

    # 7. optimize at learning rate
    learning_rate = 0.1 if step < 10000 else 0.01#lrs[step]
    for p in parameters:
        if p.grad is not None:
            p.data -= learning_rate * p.grad

    #lr_history.append(lre[step])
    lr_history.append(learning_rate)
    loss_history.append(loss.item())

loss.item()

#plt.plot(loss_history)


running std: tensor([[1.4871, 1.9450, 1.4102, 1.4219, 1.4705, 1.6286, 1.4367, 1.5882, 1.7893,
         1.5011, 1.7213, 2.1636, 1.5255, 1.3043, 1.4090, 1.5294, 1.0403, 1.4098,
         1.3027, 1.5464, 1.4792, 1.1448, 1.2589, 1.6350, 1.4284, 1.7406, 1.6876,
         1.7311, 1.3893, 1.5465, 1.6172, 1.2674, 1.6777, 1.9267, 1.3900, 1.3018,
         2.0988, 1.4486, 1.6210, 1.4534, 1.4558, 1.1353, 2.0308, 2.6108, 1.7383,
         1.4486, 1.7039, 1.8708, 1.9542, 1.6369, 1.9299, 1.3069, 2.0266, 1.7531,
         1.3692, 1.4445, 1.6263, 1.7545, 1.5979, 1.8008, 1.6953, 2.3595, 1.9678,
         1.6946, 1.3937, 1.3948, 1.6609, 2.0342, 2.0642, 1.5163, 1.3393, 2.5077,
         2.0682, 1.8187, 1.4412, 2.0803, 1.6368, 1.8314, 1.4737, 1.1919, 1.1220,
         1.9363, 1.4210, 1.4163, 1.2063, 1.6567, 1.7846, 1.5666, 1.5965, 1.7464,
         1.7193, 1.4389, 1.8231, 2.2746, 1.6453, 1.4081, 2.1907, 1.6740, 1.1549,
         1.4020, 1.1160, 1.5280, 1.4690, 1.6290, 1.3506, 1.5192, 1.6466, 2.4253,
         1.1918

2.070830821990967

In [56]:
@torch.no_grad()
def evaluate_loss(X, Y):
    h_input = C[X].view(-1, in_dim)
    h_output_preact = h_input @ w1 #Remove b1 for batch norm
    #Use running mean and std for batch norm
    h_output_bn = bn_gain * (h_output_preact - bn_mean_running) / bn_std_running + bn_bias
    h_output = torch.tanh(h_output_bn)
    logits = h_output @ w2 + b2
    return F.cross_entropy(logits, Y).item()

print(f"Dev Loss: {evaluate_loss(Xdev, Ydev):.4f}")
print(f"Test Loss: {evaluate_loss(Xtest, Ytest):.4f}")


Dev Loss: 2.3791
Test Loss: 2.3965
